# Online Retail II - Data Cleaning

## Objective
Clean and prepare the Online Retail II dataset for SQL analysis and Power BI visualization.

### Tasks
- Load the raw dataset
- Standardize column names
- Convert data types
- Handle missing values
- Investigate duplicates
- Investigate returns and cancellations
- Save the cleaned dataset

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/online_retail_II.csv")

In [3]:
# Standardize the column names
df.columns = [
    "invoice",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "price",
    "customer_id",
    "country"
]

### Create the cleaning copy 

In [4]:
clean_df = df.copy()

In [6]:
clean_df["invoice_date"].dtype

dtype('O')

#### Convert the column data type

In [7]:
clean_df["invoice_date"] = pd.to_datetime(clean_df["invoice_date"])
#pd --> take the text values and convert them into real datetime objects

In [8]:
clean_df["invoice_date"].dtype

dtype('<M8[ns]')

### Investigate missing values
initial decisions:
- description --> 4,382 (will decide wether to remove or fill them)
- customer_id --> 243,007 (keeping bc they represent anonymous customer)

In [9]:
clean_df.isnull().sum()

invoice              0
stock_code           0
description       4382
quantity             0
invoice_date         0
price                0
customer_id     243007
country              0
dtype: int64

In [10]:
clean_df[clean_df["description"].isnull()]

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
1060783,581199,84581,NaN,-2,2011-12-07 18:26:00,0.0,NaN,United Kingdom
1060787,581203,23406,NaN,15,2011-12-07 18:31:00,0.0,NaN,United Kingdom
1060793,581209,21620,NaN,6,2011-12-07 18:35:00,0.0,NaN,United Kingdom
1062442,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom


In [11]:
#to check if all rows with a missing decription also have a price of 0
clean_df[clean_df["description"].isnull()]["price"].value_counts()

price
0.0    4382
Name: count, dtype: int64

In [13]:
# inspect invoices to check if they contain normal products 
clean_df[clean_df["invoice"] == "489521"]

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom


In [14]:
# check how many unique invoices are affected
clean_df[clean_df["description"].isnull()]["invoice"].nunique()

4275

Based on the evidence so far.
the records appear to be non standard transactions because they have no description, a unit price of zero, no customer ID, and mostly as standalone invoices. They are unlikely to represent normal retail sales.

In [15]:
clean_df[clean_df["description"].isnull()]["invoice"].head(20)

470      489521
3114     489655
3161     489659
3731     489781
4296     489806
4566     489821
6378     489882
6555     489898
6576     489901
6581     489903
7204     490015
7205     490016
7559     490055
8553     490084
9249     490123
9667     490146
10236    490150
10508    490165
11973    490354
12237    490379
Name: invoice, dtype: object

In [18]:
#how many rows in the entire dataset have a price of 0
(clean_df["price"]== 0).sum()

6202

### Investigation Notes

- Missing descriptions: 4,382 rows.
- All missing descriptions have a price of 0.
- There are 6,202 zero-price records in total.
- Therefore, zero price is **not unique** to missing descriptions and cannot alone justify removing rows.

In [ ]:
# will investigate the 1820 rows (6202-4382), that does have a description
clean_df[
    (clean_df["price"] == 0) &
    (clean_df["description"].notnull())
]

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
1060797,581213,22576,check,-30,2011-12-07 18:38:00,0.0,NaN,United Kingdom
1062371,581226,23090,missing,-338,2011-12-08 09:56:00,0.0,NaN,United Kingdom
1063965,581406,46000M,POLYESTER FILLER PAD 45x45cm,240,2011-12-08 13:58:00,0.0,NaN,United Kingdom
1063966,581406,46000S,POLYESTER FILLER PAD 40x40cm,300,2011-12-08 13:58:00,0.0,NaN,United Kingdom


In [21]:
clean_df[
    (clean_df["description"].isnull())&
    (clean_df["quantity"]>0)
]

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom
6555,489898,79323G,NaN,954,2009-12-03 09:40:00,0.0,NaN,United Kingdom
6581,489903,21166,NaN,48,2009-12-03 09:57:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
1059173,581103,22689,NaN,4,2011-12-07 11:58:00,0.0,NaN,United Kingdom
1060787,581203,23406,NaN,15,2011-12-07 18:31:00,0.0,NaN,United Kingdom
1060793,581209,21620,NaN,6,2011-12-07 18:35:00,0.0,NaN,United Kingdom
1062442,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom


In [22]:
clean_df[
    (clean_df["description"].isnull())&
    (clean_df["price"]!=0)
]

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country


I investigated all records with missing descriptions and found that every one of them also had a unit price of 0 and no customr ID. These records did not represent normal customer sales and would distort the sales analysis, so i will exclude them from the cleaned dataset.

In [23]:
# to measure the impact --> see how many rows we r about to delete
removal_condition = (
    clean_df["description"].isnull()&
    (clean_df["price"]==0) &
    (clean_df["customer_id"].isnull())
)
removal_condition.sum()

4382

## Investigation: Missing Descriptions

### Findings

- Identified **4,382** rows with missing product descriptions.
- Every one of these rows also had:
  - a unit price of **0**,
  - a missing customer ID,
  - and represented non-standard transactions rather than customer purchases.

### Decision

These records were excluded from the cleaned dataset because the objective of this project is to analyze customer sales. Including inventory or administrative records would distort sales metrics such as revenue, product performance, and customer behavior.

In [24]:
clean_df.shape

(1067371, 8)

In [25]:
#keepign everything that is NOT a removal condition
clean_df=clean_df[~removal_condition]

In [26]:
clean_df.shape

(1062989, 8)

In [27]:
#verify cleaning
clean_df["description"].isnull().sum()

0

In [28]:
removal_condition.sum()

4382